Line Charts & Slopegraphs: CO2 Emissions

In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [5]:
print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


Task 1 — Multi-Series Line Chart with Highlight


In [6]:
import pandas as pd
import plotly.graph_objects as go

# Filter Asia
asia_df = df[df['Region'] == 'Asia']

# Choose highlight country (you can change this)
highlight_country = 'China'

# Create figure
fig = go.Figure()

# Plot all countries
for country in asia_df['Country'].unique():
    country_data = asia_df[asia_df['Country'] == country]

    # Highlight logic
    if country == highlight_country:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#E63946', width=3),
            name=country
        ))

        # End label
        fig.add_trace(go.Scatter(
            x=[country_data['Year'].max()],
            y=[country_data['CO2_Mt'].iloc[-1]],
            mode='text',
            text=[country],
            textposition='middle right',
            showlegend=False
        ))
    else:
        fig.add_trace(go.Scatter(
            x=country_data['Year'],
            y=country_data['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1),
            showlegend=False
        ))

# Layout styling (clean lecture style)
fig.update_layout(
    title="China’s CO₂ Growth Dominates Asia’s Emissions Landscape — Industrial Expansion Drives the Divergence",
    xaxis_title="Year",
    yaxis_title="CO₂ Emissions (Mt)",
    template="plotly_white",
    font=dict(family="Arial"),
    height=600,
    showlegend=False
)

fig.show()

Task 2 — Slopegraph: Regional Change 2000 vs 2022


In [7]:
import pandas as pd
import plotly.graph_objects as go

# Aggregate regional means
region_year = (
    df.groupby(['Region', 'Year'])['CO2_Mt']
      .mean()
      .reset_index()
)

# Filter endpoints
df_2000 = region_year[region_year['Year'] == 2000]
df_2022 = region_year[region_year['Year'] == 2022]

# Merge for slopegraph structure
slope_df = pd.merge(
    df_2000, df_2022,
    on='Region',
    suffixes=('_2000', '_2022')
)

# Determine direction (increase vs decrease)
slope_df['change'] = slope_df['CO2_Mt_2022'] - slope_df['CO2_Mt_2000']

fig = go.Figure()

for _, row in slope_df.iterrows():
    color = "#2A9D8F" if row['change'] > 0 else "#E76F51"

    # Line
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[row['CO2_Mt_2000'], row['CO2_Mt_2022']],
        mode='lines+markers',
        line=dict(color=color, width=3),
        showlegend=False
    ))

    # Left label (2000)
    fig.add_trace(go.Scatter(
        x=[2000],
        y=[row['CO2_Mt_2000']],
        mode='text',
        text=[f"{row['Region']} {row['CO2_Mt_2000']:.1f}"],
        textposition='middle left',
        showlegend=False
    ))

    # Right label (2022)
    fig.add_trace(go.Scatter(
        x=[2022],
        y=[row['CO2_Mt_2022']],
        mode='text',
        text=[f"{row['CO2_Mt_2022']:.1f}"],
        textposition='middle right',
        showlegend=False
    ))

# Styling (clean lecture style)
fig.update_layout(
    title="Asia and North America Drive CO₂ Increases, While Europe Shows Stabilization",
    xaxis=dict(
        tickmode='array',
        tickvals=[2000, 2022]
    ),
    yaxis=dict(
        showticklabels=False
    ),
    template="plotly_white",
    font=dict(family="Arial"),
    height=600,
    showlegend=False
)

fig.show()